<img
  src="https://ai-fullstack-assets.s3.eu-west-3.amazonaws.com/M03-Data_Infrastructure/M03-D01-finpulse-logo.png"
  alt="FinPulse logo"
/>

# FinPulse Data Pipeline

You work as a junior data engineer at FinPulse, a small financial research team that tracks cryptocurrency markets. The team needs fresh prices when a report is due, but right now someone pulls them by hand.

You already have a working `fetch_prices.py` collector that writes `prices.csv`. Your manager now wants that script to run automatically every morning on a remote server, even when your laptop is closed.

You already know how to launch an EC2 instance, connect with SSH, copy files with `scp`, and schedule commands with `cron`. Keep `fetch_prices.py` on your laptop, you will upload it in Task 1.


## Task 1: Upload the script and install its dependency

Use `scp` to upload `fetch_prices.py` from your laptop to the home directory of your EC2 instance. Then SSH in and install `requests`.

<Note type="hint">
A typical `scp` upload looks like this: `scp -i <path-to-key.pem> fetch_prices.py ec2-user@<public-ip>:~`.
After uploading, SSH in with `ssh -i <path-to-key.pem> ec2-user@<public-ip>`.
Your instance public IP (or DNS) is in the EC2 console under Instances, then your instance, then Public IPv4 address or Public IPv4 DNS.
</Note>

<Note type="warning">
Make sure your EC2 instance is in "running" state before uploading. If you stopped and restarted the instance, the public IP may have changed.
</Note>

<Note type="tip" title="Reconnecting after stop and start">
When an instance is restarted, the public IPv4 address and public DNS can change. Grab the new value in the AWS console (EC2, Instances, select your instance, then Public IPv4 DNS or Public IPv4 address), then reconnect with `ssh -i <path-to-key.pem> ec2-user@<public-dns-or-ip>`.
</Note>

<Note type="tip" title="If pip3 is missing">
On some EC2 images, `pip3` is not installed by default. Inside the instance, install it with `sudo dnf install -y python3-pip`, then use `python3 -m pip install --user requests`.
</Note>

<Note type="tip" title="Verify before installing">
Run `ls ~` after the `scp` step to confirm `fetch_prices.py` arrived on the instance before installing the dependency.
</Note>

Run these commands in your terminal.

```bash
# On your laptop, upload the script
scp -i ___ fetch_prices.py ec2-user@___:~
# TODO: path to your .pem file | your instance public IP (or DNS)

# Verify the file arrived
ssh -i ___ ec2-user@___ "ls ~"

# SSH into the instance
ssh -i ___ ec2-user@___

# Once inside, install pip if needed
sudo dnf install -y python3-pip

# Install the dependency
python3 -m pip install --user requests
```


## Task 2: Run the script manually and verify output

SSH into your instance and run `python3 fetch_prices.py`. Check that `prices.csv` was created in the home directory.

<Note type="hint">
SSH in first, then run `python3 fetch_prices.py`. After it finishes, run `cat ~/prices.csv` to see the file contents.
Three rows (bitcoin, ethereum, solana) with prices and timestamps mean it worked.
</Note>

<Note type="important">
If the script fails here, fix it before moving to Task 3. Scheduling a broken script means it fails on a schedule and the data will be missing when someone needs it.
</Note>

Run these commands in your terminal.

```bash
# SSH into your instance
ssh -i ___ ec2-user@___

# Once inside, run the script
python3 ___
# TODO: script filename

# Verify the output file exists and contains data
cat ~/prices.csv
```


## Task 3: Schedule the job with cron

Open `crontab -e` and add an entry that runs `fetch_prices.py` at 6:00 AM UTC every day, appending output and errors to `~/prices.log`. Verify the entry is registered with `crontab -l`.

<Note type="hint">
The entry format is: `<minute> <hour> * * * python3 ~/fetch_prices.py >> ~/prices.log 2>&1`.
For 6:00 AM UTC, minute is `0` and hour is `6`.
The `>> ~/prices.log 2>&1` part appends both stdout and stderr to the log so you can inspect past runs.
</Note>

<Note type="warning" title="Cron uses the instance clock">
Cron uses the time on the EC2 instance, which is often set to UTC by default. Before choosing your schedule, run `date` inside the instance so you know what time zone you are targeting.
If you are scheduling for Paris time, convert it to UTC first. Paris is UTC+2 in summer (CEST) and UTC+1 in winter (CET).
</Note>

<Note type="tip" title="Command spelling and missing crontab">
The editor command is `crontab` with two `t` characters. A common typo is `cronttab`, which will show as command not found.
If you see `crontab: command not found` on a fresh Amazon Linux 2023 instance, install the scheduler package and start the service: `sudo dnf install -y cronie`, then `sudo systemctl enable --now crond`, then try `crontab -e` again.
</Note>

<Note type="tip" title="If crontab opens in vi">
Type `i` to enter insert mode, paste your entry, press `Esc`, then type `:wq` and Enter to save.
To use nano instead, run `export EDITOR=nano` before `crontab -e`.
</Note>

Run these commands in your terminal (inside your EC2 instance).

```bash
# Open the cron editor
crontab -e

# Entry to add, fill in the two time fields
___ ___ * * * python3 ~/fetch_prices.py >> ~/prices.log 2>&1
# TODO: minute (0 to 59) | hour (0 to 23, UTC)

# After saving, verify the entry was registered
crontab -l

# After the scheduled time, confirm the job ran
ls -l ~/prices.csv
ls -l ~/prices.log

# Inspect the latest log lines (errors show up here too)
tail -n 50 ~/prices.log

# View the CSV content
cat ~/prices.csv

# Optional: nicer CSV display (if `column` is available)
column -s, -t < ~/prices.csv
```

<Note type="important" title="Stop your instance">
Go to the EC2 console and stop (not terminate) your instance when you are done for the day.
A running instance charges by the hour even when idle. Stopping it preserves your files and cron settings for next time.
</Note>

### Stretch

Modify the cron entry to run every hour (`0 * * * *`). Update `fetch_prices.py` to append rows to the CSV instead of overwriting it (`mode="a"` in `to_csv()`, with `header=False` after the first write). After a few hours, check `prices.csv` on the instance and observe the price history accumulating.
